# Fine-tuning SLM - Chatbot Tim Legal (PGABL)
**Nama:** Stanley Nathanael Wijaya

Notebook ini melatih ulang (fine-tune) sebuah Small Language Model (SLM) — **Qwen2.5-1.5B-Instruct**
— menggunakan teknik **QLoRA** di atas dataset instruksi berbahasa Indonesia
[`Ichsan2895/alpaca-gpt4-indonesian`](https://huggingface.co/datasets/Ichsan2895/alpaca-gpt4-indonesian),
sehingga model dapat dipakai sebagai otak dari chatbot RAG Tim Legal.

**Rekomendasi environment:** Google Colab / Kaggle dengan GPU T4 (16GB) — aktifkan
`Runtime > Change runtime type > T4 GPU` sebelum menjalankan notebook ini.

Struktur notebook:
1. Instalasi dependency
2. Load & mapping dataset ke Chat Template (ChatML)
3. Load model dasar dengan QLoRA (4-bit, double quantization) + LoRA adapter
4. Split train/validation & 2 eksperimen hyperparameter (Skilled)
5. Training penuh SFTTrainer >= 800 steps
6. Push model hasil fine-tuning ke Hugging Face Hub (`merged_16bit`)


## 1. Instalasi Dependency

In [ ]:
%%capture
import importlib
if importlib.util.find_spec("unsloth") is None:
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --no-deps trl peft accelerate bitsandbytes xformers
!pip install -q datasets wandb

In [ ]:
import os
import random

import numpy as np
import torch

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## 2. Autentikasi Hugging Face & Weights & Biases

Token **tidak pernah ditulis langsung di notebook** — gunakan `getpass` supaya nilainya tidak
tersimpan sebagai plaintext di sel kode maupun output.


In [ ]:
from getpass import getpass
from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("Masukkan Hugging Face Write Token: ")
login(token=HF_TOKEN)

HF_USERNAME = os.environ.get("HF_USERNAME") or input("Masukkan username Hugging Face kamu: ")
FT_REPO_ID = f"{HF_USERNAME}/qwen2.5-1.5b-legal-chatbot-id"
print("Model fine-tuning akan di-push ke:", FT_REPO_ID)

In [ ]:
try:
    import wandb

    WANDB_TOKEN = os.environ.get("WANDB_API_KEY") or getpass(
        "Masukkan WandB API Key (kosongkan + Enter untuk skip logging): "
    )
    if WANDB_TOKEN:
        wandb.login(key=WANDB_TOKEN)
        REPORT_TO = "wandb"
    else:
        REPORT_TO = "none"
except Exception as exc:  # noqa: BLE001
    print("WandB tidak digunakan:", exc)
    REPORT_TO = "none"

## 3. Load & Mapping Dataset ke Chat Template

Dataset mentah `Ichsan2895/alpaca-gpt4-indonesian` memiliki kolom `input` (berisi instruksi) dan
`output` (jawaban). Kita normalisasi dulu ke skema Alpaca standar (`instruction`, `input`, `output`)
sebelum melakukan mapping ke Chat Template ChatML yang didukung Unsloth.


In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")
print(raw_dataset)
print(raw_dataset.column_names)

In [ ]:
# Normalisasi skema -> format Alpaca standar (instruction, input, output)
def normalize_to_alpaca(example):
    return {
        "instruction": example["input"],
        "input": "",
        "output": example["output"],
    }


if "instruction" not in raw_dataset.column_names:
    raw_dataset = raw_dataset.map(normalize_to_alpaca, remove_columns=raw_dataset.column_names)

print("Contoh dataset yang BELUM di-mapping ke chat template:")
print(raw_dataset[0])

In [ ]:
from unsloth.chat_templates import get_chat_template

max_seq_length = 2048

SYSTEM_PROMPT = (
    "Kamu adalah asisten AI internal Tim Legal perusahaan. Jawablah pertanyaan "
    "seputar peraturan dan perundang-undangan secara akurat, ringkas, dan dalam Bahasa Indonesia."
)


def formatting_prompts_func(examples):
    texts = []
    for instruction, inp, output in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        user_content = instruction if not inp else f"{instruction}\n\n{inp}"
        convo = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output},
        ]
        texts.append(tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

Load tokenizer sementara (dari base model) khusus untuk memformat dataset lebih dulu.

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer = get_chat_template(tokenizer, chat_template="chatml")

dataset = raw_dataset.map(formatting_prompts_func, batched=True)

print("Contoh dataset yang SUDAH di-mapping ke chat template (dengan token spesial):")
print(dataset[0]["text"])

## 4. Load Model dengan QLoRA (4-bit + Double Quantization) dan LoRA Adapter

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=max_seq_length,
    dtype=None,  # auto-detect (fp16 di GPU lama, bf16 di GPU baru)
    load_in_4bit=True,  # QLoRA: quantize base model ke 4-bit (NF4 + double quantization)
)
tokenizer = get_chat_template(tokenizer, chat_template="chatml")

# Pastikan double quantization benar-benar aktif
print("Quantization config:", model.config.quantization_config)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    # LoRA ditempatkan pada KEDUA komponen komputasi utama:
    #   - Multi-Head Attention: q_proj, k_proj, v_proj, o_proj
    #   - Feed Forward Network : gate_proj, up_proj, down_proj
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()

## 5. Train/Validation Split & Eksperimen Hyperparameter (Skilled)

In [ ]:
split_dataset = dataset.train_test_split(test_size=0.05, seed=SEED)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]
print(f"Train: {len(train_dataset)} | Validation: {len(eval_dataset)}")

In [ ]:
from trl import SFTConfig, SFTTrainer


def build_trainer(run_name, learning_rate, lora_r_note, max_steps, per_device_train_batch_size=2,
                   gradient_accumulation_steps=4, warmup_steps=10):
    args = SFTConfig(
        output_dir=f"outputs/{run_name}",
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=learning_rate,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        report_to=REPORT_TO,
        run_name=run_name,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        packing=False,
    )
    return SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=args,
    )

### Eksperimen 1 vs Eksperimen 2

Dua kombinasi hyperparameter berbeda dijalankan dalam jumlah step yang lebih singkat (300 steps)
khusus untuk **membandingkan kurva loss** dan memilih kombinasi terbaik sebelum commit ke training
penuh (>= 800 steps) pada sel berikutnya.


In [ ]:
trainer_exp1 = build_trainer(
    run_name="exp1_lr2e-4",
    learning_rate=2e-4,
    lora_r_note="r=16, lr=2e-4",
    max_steps=300,
)
stats_exp1 = trainer_exp1.train()

In [ ]:
# Reload model bersih untuk eksperimen kedua (hyperparameter berbeda: LR lebih kecil + warmup lebih panjang)
model = FastLanguageModel.get_peft_model(
    FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=max_seq_length, dtype=None, load_in_4bit=True,
    )[0],
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

trainer_exp2 = build_trainer(
    run_name="exp2_lr5e-5",
    learning_rate=5e-5,
    lora_r_note="r=16, lr=5e-5",
    max_steps=300,
    warmup_steps=30,
)
stats_exp2 = trainer_exp2.train()

In [ ]:
import matplotlib.pyplot as plt


def get_eval_loss_curve(trainer):
    logs = trainer.state.log_history
    steps = [l["step"] for l in logs if "eval_loss" in l]
    losses = [l["eval_loss"] for l in logs if "eval_loss" in l]
    return steps, losses


steps1, loss1 = get_eval_loss_curve(trainer_exp1)
steps2, loss2 = get_eval_loss_curve(trainer_exp2)

plt.figure(figsize=(7, 5))
plt.plot(steps1, loss1, marker="o", label="Eksperimen 1 (lr=2e-4)")
plt.plot(steps2, loss2, marker="o", label="Eksperimen 2 (lr=5e-5)")
plt.xlabel("Step")
plt.ylabel("Eval Loss")
plt.title("Perbandingan Kurva Eval Loss Antar Eksperimen")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

best_lr = 2e-4 if (loss1 and min(loss1) <= min(loss2 or [float("inf")])) else 5e-5
print(f"Kombinasi hyperparameter terbaik (eval_loss terendah tanpa overfitting): learning_rate={best_lr}")

## 6. Training Penuh (>= 800 steps) dengan Hyperparameter Terbaik

In [ ]:
# Reload model bersih sekali lagi untuk training final yang akan di-push ke Hugging Face
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=max_seq_length, dtype=None, load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="chatml")
model = FastLanguageModel.get_peft_model(
    base_model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

final_trainer = build_trainer(
    run_name="final_run",
    learning_rate=best_lr,
    lora_r_note="final",
    max_steps=800,
    warmup_steps=20,
)
final_stats = final_trainer.train()
print(final_stats)

## 7. Uji Cepat Model Hasil Fine-tuning

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Apa itu Perjanjian Kerja Waktu Tertentu (PKWT)?"},
]
inputs = tokenizer.apply_chat_template(
    test_prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

## 8. Push Model ke Hugging Face Hub (`merged_16bit`)

Model hasil fine-tuning (adapter LoRA digabung/merge ke base model dalam presisi 16-bit) diunggah
ke Hugging Face agar dapat dipanggil kembali di notebook RAG.


In [ ]:
model.push_to_hub_merged(
    FT_REPO_ID,
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN,
)
print("Model berhasil di-push ke:", f"https://huggingface.co/{FT_REPO_ID}")

In [ ]:
with open("link_huggingface.txt", "a") as f:
    f.write(f"Fine-tuned model: https://huggingface.co/{FT_REPO_ID}\n")
print(open("link_huggingface.txt").read())